# mBART — NLP Robot Command Parser

Training, evaluation, ASR and HuRIC evaluation for the **mBART-large-50** model.
Results are saved to `results/mbart/` and checkpoints to `checkpoints/mbart/final`.

Run this notebook independently — it loads data and trains from scratch.

## 0. Setup
Prepares the environment. The repository and the SCAN dataset are cloned from GitHub. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [ ]:
# Clone repo and move into it 
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

In [ ]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)       
torch.backends.cudnn.deterministic = True  
torch.backends.cudnn.benchmark = False 

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {cfg["model"]["name"]}')


## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration. 

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'
split = cfg['data']['scan_split'] 

train_data, test_data = load_scan(
    split     = split,
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language : {language}')
print(f'Train examples : {len(train_data)}')
print(f'Test  examples : {len(test_data)}')
print(f'First example  : {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer = get_tokenizer(cfg['model']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test' : to_hf_dataset(test_data),
})

tokenized_dataset = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset)

## 3. Config for mBART


In [ ]:
model_cfg_mbart = cfg["model_mbart"]
training_cfg_mbart = cfg["training_mbart"]

print("Model:", model_cfg_mbart["name"])
print("Out dir:", training_cfg_mbart["output_dir"])

## 4. Training

mBART-large-50 is a multilingual seq2seq model pretrained on 50 languages (including Serbian). Results are saved in a `results/mbart/`.

In [ ]:
#  Load mBART tokeniser + model 
from src.models.mbart_model import load_mbart_model

model_mbart, tokenizer_mbart = load_mbart_model(model_cfg_mbart["name"], DEVICE)

In [ ]:
#  Tokenise dataset for mBART 
# get_tokenizer detects "mbart" in the name and returns MBart50TokenizerFast
from src.data.preprocess import to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

raw_dataset_mbart = DatasetDict({
    "train": to_hf_dataset(train_data),
    "test": to_hf_dataset(test_data),
})

tokenized_dataset_mbart = tokenize_dataset(
    raw_dataset_mbart,
    tokenizer_mbart,
    prefix = model_cfg_mbart["prefix"],
    max_input_len = model_cfg_mbart["max_input_len"],
    max_target_len = model_cfg_mbart["max_target_len"],
)

In [ ]:
# Train mBART 
from src.training.trainer import build_trainer

trainer_mbart = build_trainer(
    model=model_mbart,
    tokenizer=tokenizer_mbart,
    tokenized_dataset=tokenized_dataset_mbart,
    cfg=cfg,
    model_key='model_mbart',
    device_fp16=(DEVICE == 'cuda'),
)

trainer_mbart.train()

In [ ]:
from src.training.trainer import get_checkpoint_dir
# Save mBART checkpoint
SAVE_PATH = get_checkpoint_dir(cfg, 'model_mbart')
model_mbart.save_pretrained(SAVE_PATH)
tokenizer_mbart.save_pretrained(SAVE_PATH)
print(f"mBART model saved to: {SAVE_PATH}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
drive_dest = f"/content/drive/MyDrive/nlp-robot-command-parser/{SAVE_PATH}"
os.makedirs(drive_dest, exist_ok=True)
shutil.copytree(SAVE_PATH, drive_dest, dirs_exist_ok=True)
print(f"Checkpoint copied to Google Drive: {drive_dest}")

## 5. Loading trained model

In [ ]:
from src.training.trainer import get_checkpoint_dir
from transformers import T5ForConditionalGeneration, T5Tokenizer
import os

LOAD_PATH = get_checkpoint_dir(cfg, 'model')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    LOAD_PATH = f"/content/drive/MyDrive/nlp-robot-command-parser/{LOAD_PATH}"
except ImportError:
    pass  

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(f'Model not found at: {LOAD_PATH}')

model_mbart = T5ForConditionalGeneration.from_pretrained(LOAD_PATH)
tokenizer_mbart = T5Tokenizer.from_pretrained(LOAD_PATH)
model_mbart = model_mbart.to(DEVICE)

print(f' Model loaded from: {LOAD_PATH}')
print(f' Device: {DEVICE}')